In [ ]:
import torch
from torch import Tensor

class CategoricalSamplingFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: Tensor):
        # Forward is only used in eager runs (not during ONNX parsing).
        # Return INT32 tensor to match plugin output type
        return torch.zeros(x.shape[0], dtype=torch.int32, device=x.device)

    @staticmethod
    def symbolic(g, x):
        # Emit ONNX node whose (domain, op_type) matches the TRT plugin creator.
        # Pass non-tensor params as attributes; tensor params as inputs.
        # Use empty namespace to match the plugin registration
        # Specify output type as INT32 using type_as parameter
        output = g.op("CategoricalSampling", x)
        # Set the output type to INT32 with 1D shape (dynamic size)
        #output.setType(x.type().with_dtype(torch.int32))
        output.setType(x.type().with_dtype(torch.int32).with_sizes([None]))
        return output

def categorical_sampling(x: Tensor) -> Tensor:
    return CategoricalSamplingFn.apply(x)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os

class LinearLT(nn.Module):
    def __init__(self, dim=768, vocab_size=2024):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim)
        self.linear = nn.Linear(dim, vocab_size)

    def forward(self, hidden_states):
        # Ensure we use the same batch dimension for both operations
        # hidden_states: (batch_size, in_dim)
        # tokens: (tokens_num, half_batch_size)
        
        # Get the half batch size from tokens shape
        half_batch_size = hidden_states.shape[0] // 2 # make this dynamic later
        tokens = torch.zeros((half_batch_size,0), dtype=torch.int).cuda()
        
        # Only use the first half of hidden_states to match half_batch_size
        hidden_states_half = hidden_states[:half_batch_size].unsqueeze(1)
        
        # Now both operations use the same batch dimension
        n_steps = 8
        for i in range(n_steps):
            token_embs = self.embedding(tokens)  # half_batch_size x tokens_num x dim
            input = torch.cat([hidden_states_half, token_embs], dim=1)
            logits = self.linear(input)  # half_batch_size x dim
            probs = F.softmax(logits[:,-1:,:], dim=-1) # half_batch_size x vocab_size
            sampled_tokens = categorical_sampling(probs) # half_batch_size
            # append sampled token to token sequence
            tokens = torch.cat([tokens, sampled_tokens.unsqueeze(1)], dim=1)
        return tokens       

In [ ]:
    # --- Configuration ---
onnx_file_path = "local_transformer.onnx"

# --- Model Initialization ---
print("Initializing models...")
#lt = ToyLocalTransformer()
lt = LinearLT()
lt.eval().cuda().half()

DUMMY_HALF_BATCH_SIZE = 2
DUMMY_TOKENS_NUM = 3
dummy_hidden_state = torch.randn((DUMMY_HALF_BATCH_SIZE * 2, 768)).cuda().half()
#dummy_tokens = torch.zeros((DUMMY_TOKENS_NUM, DUMMY_HALF_BATCH_SIZE), dtype=torch.int).cuda()

print(lt(dummy_hidden_state).shape)

# --- ONNX Export ---
print(f"\nAttempting to export the model to '{onnx_file_path}'...")

torch.onnx.export(
    lt, 
    (dummy_hidden_state), 
    onnx_file_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['hidden_states'],
    output_names=['tokens'],
    dynamic_axes={
        'hidden_states': {0: 'batch_size'},
        'tokens': {0: 'half_batch_size'}
    }
)
print(f"File saved at: {os.path.abspath(onnx_file_path)}")

In [ ]:
#!pip install onnxruntime


In [ ]:
# import onnxruntime as ort
# import numpy as np

# ort_session = ort.InferenceSession(onnx_file_path, providers=['CPUExecutionProvider'])

# bs = 2
# num_tok = 8
# hs = np.random.randn(bs * 2, 16192).astype(np.float16)
# toks = np.zeros((num_tok, bs), dtype=np.int32)
# outputs = ort_session.run(None, {"hidden_states": hs, "tokens": toks})
# print(outputs[0].shape)

In [ ]:
# --fp16
! CUDA_VISIBLE_DEVICES=0 /usr/local/tensorrt/targets/x86_64-linux-gnu/bin/trtexec --fp16 \
    --onnx=/code/tensorrt_llm/local_transformer.onnx \
    --saveEngine=/code/tensorrt_llm/local_transformer.trt \
    --minShapes=hidden_states:2x768  \
    --optShapes=hidden_states:8x768 \
    --maxShapes=hidden_states:32x768 \
     --plugins=/code/tensorrt_llm/build/libcategorical_sampling_plugin.so

    #--verbose